# 12 — Incentive vs Quality (RQ8)

**Research Question:** Does financial incentive misalignment degrade service quality?

Hospitals that extract more revenue from unnecessary or inflated billing may show worse
quality outcomes. We construct a waste index per hospital and correlate it with quality
metrics — within fair comparability groups.

**Report:** `reports/12_incentive_quality.md`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from shared import (
    load_kidney, load_hospital_tags, load_cnes_enriched,
    setup_plot_style, save_plot, save_metrics,
    CATEGORY_MAP, PROC_NAMES,
)

setup_plot_style()
kidney = load_kidney()
tags = load_hospital_tags()
cnes = load_cnes_enriched()
print(f"Loaded {len(kidney):,} admissions, {len(tags)} hospitals")

Loaded 206,500 admissions, 510 hospitals


## 1. Construct Hospital Waste Index

Three components:
1. **Diagnostic admission rate** — % of admissions that are diagnostic imaging
2. **Excess LOS** — mean LOS minus peer-group median LOS
3. **Emergency rate** — higher emergency rate may indicate unnecessary ER admissions

Each component is z-scored within the hospital's comparability group, then averaged.

In [2]:
MIN_ADMISSIONS = 30

# Hospital-level metrics
hosp = kidney.groupby("CNES").agg(
    n=pd.NamedAgg(column="CNES", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
    mortality_rate=pd.NamedAgg(column="MORTE", aggfunc="mean"),
    mean_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="mean"),
    total_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="sum"),
    diagnostic_rate=pd.NamedAgg(column="proc_category",
        aggfunc=lambda x: (x == "DIAGNOSTIC").mean()),
    emergency_rate=pd.NamedAgg(column="is_emergency", aggfunc="mean"),
    long_stay_rate=pd.NamedAgg(column="DIAS_PERM",
        aggfunc=lambda x: (x > 7).mean()),
    surgical_mortality=pd.NamedAgg(column="MORTE",
        aggfunc=lambda x: x[kidney.loc[x.index, "proc_category"] == "SURGICAL"].mean()
            if (kidney.loc[x.index, "proc_category"] == "SURGICAL").any() else np.nan),
    n_surgical=pd.NamedAgg(column="proc_category",
        aggfunc=lambda x: (x == "SURGICAL").sum()),
    surgical_los=pd.NamedAgg(column="DIAS_PERM",
        aggfunc=lambda x: x[kidney.loc[x.index, "proc_category"] == "SURGICAL"].mean()
            if (kidney.loc[x.index, "proc_category"] == "SURGICAL").any() else np.nan),
).reset_index()

hosp = hosp[hosp["n"] >= MIN_ADMISSIONS]

# Join comparability group
hosp = hosp.merge(tags[["CNES", "comparability_group"]], on="CNES", how="left")

# Compute peer-group median LOS
group_medians = hosp.groupby("comparability_group")["mean_los"].median().rename("group_median_los")
hosp = hosp.merge(group_medians, on="comparability_group", how="left")
hosp["excess_los"] = hosp["mean_los"] - hosp["group_median_los"]

# Z-score each component within comparability group
def zscore_within_group(df, col):
    return df.groupby("comparability_group")[col].transform(
        lambda x: (x - x.mean()) / max(x.std(), 0.001)
    )

hosp["z_diagnostic"] = zscore_within_group(hosp, "diagnostic_rate")
hosp["z_excess_los"] = zscore_within_group(hosp, "excess_los")
hosp["z_emergency"] = zscore_within_group(hosp, "emergency_rate")

# Waste index = average of z-scores
hosp["waste_index"] = (hosp["z_diagnostic"] + hosp["z_excess_los"] + hosp["z_emergency"]) / 3

# Quartiles
hosp["waste_quartile"] = pd.qcut(hosp["waste_index"], q=4, labels=["Q1 (low)", "Q2", "Q3", "Q4 (high)"])

print(f"Hospitals with >={MIN_ADMISSIONS} admissions: {len(hosp)}")
print(f"Waste index range: {hosp['waste_index'].min():.2f} to {hosp['waste_index'].max():.2f}")
print(f"\nWaste index components correlation:")
print(hosp[["z_diagnostic", "z_excess_los", "z_emergency"]].corr().to_string())

Hospitals with >=30 admissions: 283
Waste index range: -2.70 to 2.20

Waste index components correlation:
              z_diagnostic  z_excess_los  z_emergency
z_diagnostic      1.000000      0.191622     0.424608
z_excess_los      0.191622      1.000000     0.224911
z_emergency       0.424608      0.224911     1.000000


## 2. Waste Index vs Quality Metrics

Do high-waste hospitals have worse outcomes?

In [3]:
quality_by_quartile = hosp.groupby("waste_quartile", observed=False).agg(
    n_hospitals=pd.NamedAgg(column="CNES", aggfunc="count"),
    n_admissions=pd.NamedAgg(column="n", aggfunc="sum"),
    mean_los=pd.NamedAgg(column="mean_los", aggfunc="mean"),
    mortality_rate=pd.NamedAgg(column="mortality_rate", aggfunc="mean"),
    long_stay_rate=pd.NamedAgg(column="long_stay_rate", aggfunc="mean"),
    mean_cost=pd.NamedAgg(column="mean_cost", aggfunc="mean"),
    diagnostic_rate=pd.NamedAgg(column="diagnostic_rate", aggfunc="mean"),
    emergency_rate=pd.NamedAgg(column="emergency_rate", aggfunc="mean"),
)

print("=== QUALITY BY WASTE QUARTILE ===")
print(quality_by_quartile.to_string())

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Mean LOS
quality_by_quartile["mean_los"].plot.bar(ax=axes[0, 0], color=["#059669", "#6B7280", "#D97706", "#DC2626"])
axes[0, 0].set_title("Mean LOS by Waste Quartile")
axes[0, 0].set_ylabel("Days")
axes[0, 0].tick_params(axis="x", rotation=0)

# Mortality
(quality_by_quartile["mortality_rate"]*100).plot.bar(ax=axes[0, 1], color=["#059669", "#6B7280", "#D97706", "#DC2626"])
axes[0, 1].set_title("Mortality Rate by Waste Quartile")
axes[0, 1].set_ylabel("Mortality (%)")
axes[0, 1].tick_params(axis="x", rotation=0)

# Long-stay rate
(quality_by_quartile["long_stay_rate"]*100).plot.bar(ax=axes[0, 2], color=["#059669", "#6B7280", "#D97706", "#DC2626"])
axes[0, 2].set_title("Long-Stay Rate by Waste Quartile")
axes[0, 2].set_ylabel("% patients >7d")
axes[0, 2].tick_params(axis="x", rotation=0)

# Mean cost
quality_by_quartile["mean_cost"].plot.bar(ax=axes[1, 0], color=["#059669", "#6B7280", "#D97706", "#DC2626"])
axes[1, 0].set_title("Mean Cost by Waste Quartile")
axes[1, 0].set_ylabel("BRL")
axes[1, 0].tick_params(axis="x", rotation=0)

# Diagnostic rate
(quality_by_quartile["diagnostic_rate"]*100).plot.bar(ax=axes[1, 1], color=["#059669", "#6B7280", "#D97706", "#DC2626"])
axes[1, 1].set_title("Diagnostic Rate by Waste Quartile")
axes[1, 1].set_ylabel("%")
axes[1, 1].tick_params(axis="x", rotation=0)

# Emergency rate
(quality_by_quartile["emergency_rate"]*100).plot.bar(ax=axes[1, 2], color=["#059669", "#6B7280", "#D97706", "#DC2626"])
axes[1, 2].set_title("Emergency Rate by Waste Quartile")
axes[1, 2].set_ylabel("%")
axes[1, 2].tick_params(axis="x", rotation=0)

fig.suptitle("Quality Metrics by Waste Index Quartile", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "waste_vs_quality", prefix="12")

# Statistical test: Q4 vs Q1
q4 = hosp[hosp["waste_quartile"] == "Q4 (high)"]
q1 = hosp[hosp["waste_quartile"] == "Q1 (low)"]
u_los, p_los = stats.mannwhitneyu(q4["mean_los"], q1["mean_los"], alternative="greater")
u_mort, p_mort = stats.mannwhitneyu(q4["mortality_rate"], q1["mortality_rate"], alternative="greater")
print(f"\nQ4 vs Q1 — LOS: Mann-Whitney U p = {p_los:.4f}")
print(f"Q4 vs Q1 — Mortality: Mann-Whitney U p = {p_mort:.4f}")

=== QUALITY BY WASTE QUARTILE ===
                n_hospitals  n_admissions  mean_los  mortality_rate  long_stay_rate   mean_cost  diagnostic_rate  emergency_rate
waste_quartile                                                                                                                  
Q1 (low)                 70         51578  1.999266        0.003242        0.029474  758.750941         0.275706        0.562390
Q2                       69         65297  2.181127        0.002971        0.039206  687.607170         0.391452        0.697665
Q3                       69         33446  2.561559        0.005181        0.038405  520.880441         0.622861        0.801844
Q4 (high)                69         53530  3.489685        0.004551        0.085937  610.096601         0.587375        0.816454


  Saved: 12_waste_vs_quality.png

Q4 vs Q1 — LOS: Mann-Whitney U p = 0.0000
Q4 vs Q1 — Mortality: Mann-Whitney U p = 0.1459


## 3. Cost vs Outcomes — Does Paying More Get Better Care?

Within peer groups, does higher mean cost per admission predict better outcomes?

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost vs LOS
axes[0].scatter(hosp["mean_cost"], hosp["mean_los"],
               s=hosp["n"]/10, alpha=0.4, c=hosp["waste_index"],
               cmap="RdYlGn_r", edgecolors="none")
r_cost_los, p_cost_los = stats.pearsonr(hosp["mean_cost"], hosp["mean_los"])
axes[0].set_title(f"Mean Cost vs Mean LOS (r={r_cost_los:.2f}, p={p_cost_los:.4f})")
axes[0].set_xlabel("Mean cost (BRL)")
axes[0].set_ylabel("Mean LOS (days)")

# Cost vs mortality
valid_mort = hosp[hosp["mortality_rate"] > 0]
axes[1].scatter(valid_mort["mean_cost"], valid_mort["mortality_rate"]*100,
               s=valid_mort["n"]/10, alpha=0.4, c=valid_mort["waste_index"],
               cmap="RdYlGn_r", edgecolors="none")
if len(valid_mort) > 5:
    r_cost_mort, p_cost_mort = stats.pearsonr(valid_mort["mean_cost"], valid_mort["mortality_rate"])
    axes[1].set_title(f"Mean Cost vs Mortality (r={r_cost_mort:.2f}, p={p_cost_mort:.4f})")
else:
    axes[1].set_title("Mean Cost vs Mortality")
axes[1].set_xlabel("Mean cost (BRL)")
axes[1].set_ylabel("Mortality (%)")

fig.suptitle("Cost vs Outcomes (color = waste index, size = volume)",
             fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "cost_vs_outcomes", prefix="12")

  Saved: 12_cost_vs_outcomes.png


## 4. Diagnostic-Heavy Hospitals — Surgical Performance

Are hospitals with >50% diagnostic admission rates also worse at surgery?
This tests whether the diagnostic admission behavior is associated with broader quality issues.

In [5]:
# Compare surgical performance: high-diagnostic vs low-diagnostic hospitals
hosp_surg = hosp[hosp["n_surgical"] >= 20].copy()
hosp_surg["high_diagnostic"] = hosp_surg["diagnostic_rate"] > 0.5

high_diag = hosp_surg[hosp_surg["high_diagnostic"]]
low_diag = hosp_surg[~hosp_surg["high_diagnostic"]]

print(f"High-diagnostic hospitals (>50% diagnostic): {len(high_diag)}")
print(f"  Surgical LOS: {high_diag['surgical_los'].mean():.2f}d")
print(f"  Surgical mortality: {high_diag['surgical_mortality'].mean()*100:.3f}%")
print(f"Low-diagnostic hospitals: {len(low_diag)}")
print(f"  Surgical LOS: {low_diag['surgical_los'].mean():.2f}d")
print(f"  Surgical mortality: {low_diag['surgical_mortality'].mean()*100:.3f}%")

if len(high_diag) >= 5 and len(low_diag) >= 5:
    u_slos, p_slos = stats.mannwhitneyu(high_diag["surgical_los"].dropna(),
                                         low_diag["surgical_los"].dropna(),
                                         alternative="greater")
    print(f"  Surgical LOS: Mann-Whitney U p = {p_slos:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = ["Low Diagnostic\n(≤50%)", "High Diagnostic\n(>50%)"]
surg_los = [low_diag["surgical_los"].mean(), high_diag["surgical_los"].mean()]
axes[0].bar(labels, surg_los, color=["#059669", "#DC2626"])
axes[0].set_title("Surgical LOS: High vs Low Diagnostic Hospitals")
axes[0].set_ylabel("Mean surgical LOS (days)")
for i, v in enumerate(surg_los):
    axes[0].text(i, v + 0.05, f"{v:.2f}d", ha="center", fontweight="bold")

surg_mort = [low_diag["surgical_mortality"].mean()*100, high_diag["surgical_mortality"].mean()*100]
axes[1].bar(labels, surg_mort, color=["#059669", "#DC2626"])
axes[1].set_title("Surgical Mortality: High vs Low Diagnostic Hospitals")
axes[1].set_ylabel("Surgical mortality (%)")
for i, v in enumerate(surg_mort):
    axes[1].text(i, v + 0.01, f"{v:.3f}%", ha="center", fontweight="bold")

fig.suptitle("Do Diagnostic-Heavy Hospitals Also Have Worse Surgery?", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "diagnostic_vs_surgical_quality", prefix="12")

High-diagnostic hospitals (>50% diagnostic): 12
  Surgical LOS: 4.36d
  Surgical mortality: 1.462%
Low-diagnostic hospitals: 148
  Surgical LOS: 2.66d
  Surgical mortality: 0.615%
  Surgical LOS: Mann-Whitney U p = 0.0306


  Saved: 12_diagnostic_vs_surgical_quality.png


## 5. Waste Index Distribution & Hospital Profile

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Waste index distribution
axes[0].hist(hosp["waste_index"], bins=30, color="#2563EB", edgecolor="white", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--", label="Mean (0)")
axes[0].set_title("Waste Index Distribution")
axes[0].set_xlabel("Waste index (z-score composite)")
axes[0].set_ylabel("Number of hospitals")
axes[0].legend()

# Scatter: waste index vs mortality (within groups)
axes[1].scatter(hosp["waste_index"], hosp["mortality_rate"]*100,
               s=hosp["n"]/10, alpha=0.4, color="#7C3AED")
r_w, p_w = stats.pearsonr(hosp["waste_index"], hosp["mortality_rate"])
axes[1].set_title(f"Waste Index vs Mortality (r={r_w:.3f}, p={p_w:.4f})")
axes[1].set_xlabel("Waste index")
axes[1].set_ylabel("Mortality rate (%)")

fig.suptitle("Waste Index Overview", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "waste_index_overview", prefix="12")

  Saved: 12_waste_index_overview.png


## Summary Metrics

In [7]:
q4_data = quality_by_quartile.loc["Q4 (high)"]
q1_data = quality_by_quartile.loc["Q1 (low)"]

metrics = {
    "hospitals_analyzed": int(len(hosp)),
    "min_admissions_threshold": MIN_ADMISSIONS,
    "waste_index_range": f"{hosp['waste_index'].min():.2f} to {hosp['waste_index'].max():.2f}",
    "q4_mean_los": round(float(q4_data["mean_los"]), 2),
    "q1_mean_los": round(float(q1_data["mean_los"]), 2),
    "los_ratio_q4_q1": round(float(q4_data["mean_los"] / max(q1_data["mean_los"], 0.01)), 2),
    "q4_mortality_pct": round(float(q4_data["mortality_rate"] * 100), 3),
    "q1_mortality_pct": round(float(q1_data["mortality_rate"] * 100), 3),
    "q4_long_stay_pct": round(float(q4_data["long_stay_rate"] * 100), 1),
    "q1_long_stay_pct": round(float(q1_data["long_stay_rate"] * 100), 1),
    "q4_diagnostic_rate_pct": round(float(q4_data["diagnostic_rate"] * 100), 1),
    "q1_diagnostic_rate_pct": round(float(q1_data["diagnostic_rate"] * 100), 1),
    "cost_los_correlation_r": round(float(r_cost_los), 3),
    "waste_mortality_correlation_r": round(float(r_w), 3),
    "high_diagnostic_hospitals": int(len(high_diag)),
    "high_diag_surgical_los": round(float(high_diag["surgical_los"].mean()), 2),
    "low_diag_surgical_los": round(float(low_diag["surgical_los"].mean()), 2),
}
save_metrics(metrics, "12_incentive_quality")

print("\n=== INCENTIVE VS QUALITY ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 12_incentive_quality.json

=== INCENTIVE VS QUALITY ===
  hospitals_analyzed: 283
  min_admissions_threshold: 30
  waste_index_range: -2.70 to 2.20
  q4_mean_los: 3.49
  q1_mean_los: 2.0
  los_ratio_q4_q1: 1.75
  q4_mortality_pct: 0.455
  q1_mortality_pct: 0.324
  q4_long_stay_pct: 8.6
  q1_long_stay_pct: 2.9
  q4_diagnostic_rate_pct: 58.7
  q1_diagnostic_rate_pct: 27.6
  cost_los_correlation_r: -0.218
  waste_mortality_correlation_r: nan
  high_diagnostic_hospitals: 12
  high_diag_surgical_los: 4.36
  low_diag_surgical_los: 2.66
